In [1]:
pip install numpy pandas matplotlib seaborn scikit-learn xgboost lightgbm shap joblib streamlit

   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   ---------------------------------------- 555.9/555.9 kB 1.3 MB/s  0:00:00

   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   ---------------------------------------- 2/2 [shap]

Note: you may need to restart the kernel to us

In [19]:
# =========================================
# 🔷 1. IMPORT ESSENTIAL LIBRARIES
# =========================================
import numpy as np
import pandas as pd

# Reproducibility
np.random.seed(42)

# Dataset size
N = 50000

# =========================================
# 🔷 2. DEMOGRAPHIC FEATURES
# =========================================
age = np.random.randint(18, 80, N)
gender = np.random.choice(['Male', 'Female'], N)

height = np.random.normal(165, 10, N)  # cm
weight = np.random.normal(70, 15, N)   # kg
bmi = weight / ((height / 100) ** 2)

# =========================================
# 🔷 3. LIFESTYLE FEATURES
# =========================================
daily_steps = np.random.randint(1000, 15000, N)

activity_level = np.where(daily_steps < 4000, 'Low',
                  np.where(daily_steps < 8000, 'Moderate', 'High'))

job_type = np.where(activity_level == 'Low', 'Sedentary', 'Active')

smoking = np.random.binomial(1, 0.3, N)

alcohol = np.random.choice(['None', 'Occasional', 'Frequent'], N, p=[0.5, 0.3, 0.2])

sleep_hours = np.clip(np.random.normal(6.5, 1.2, N), 3, 10)

# =========================================
# 🔷 4. CLINICAL FEATURES
# =========================================
systolic_bp = np.random.normal(110 + (age * 0.4) + (bmi * 0.6), 10, N)
diastolic_bp = np.random.normal(70 + (age * 0.25), 8, N)

fasting_sugar = np.random.normal(85 + (bmi * 1.8), 15, N)
hba1c = np.random.normal(5 + (fasting_sugar / 55), 0.8, N)

hemoglobin = np.random.normal(14 - (0.02 * age), 1.5, N)

cholesterol = np.random.normal(170 + (bmi * 2.5) + (smoking * 20), 25, N)

creatinine = np.random.normal(0.8 + (age * 0.012), 0.2, N)

heart_rate = np.random.normal(75 - (daily_steps / 6000), 10, N)

# =========================================
# 🔷 5. ADVANCED FEATURE: VISCERAL FAT 🔥
# =========================================
visceral_fat = (
    0.6 * bmi +
    0.04 * age -
    0.0006 * daily_steps +
    np.random.normal(0, 2, N)
)

# =========================================
# 🔷 6. HELPER FUNCTION
# =========================================
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =========================================
# 🔷 7. DISEASE PROBABILITY MODELS
# =========================================

# Diabetes
diabetes_prob = sigmoid(
    0.03 * fasting_sugar +
    0.9 * hba1c +
    0.05 * bmi +
    0.05 * visceral_fat -
    0.0003 * daily_steps -
    6.5
)

# Hypertension
hypertension_prob = sigmoid(
    0.04 * systolic_bp +
    0.03 * diastolic_bp +
    0.02 * age +
    0.04 * visceral_fat +
    0.5 * smoking -
    8
)

# Heart Disease
heart_disease_prob = sigmoid(
    0.035 * cholesterol +
    0.04 * systolic_bp +
    0.7 * smoking +
    0.04 * visceral_fat +
    0.02 * age -
    0.0002 * daily_steps -
    9
)

# Anemia
anemia_prob = sigmoid(
    -0.9 * hemoglobin +
    0.015 * age +
    0.4 * (gender == 'Female').astype(int) -
    2
)

# Kidney Disease
kidney_disease_prob = sigmoid(
    2.2 * creatinine +
    0.02 * age +
    0.6 * diabetes_prob +
    0.5 * hypertension_prob -
    3.5
)

# =========================================
# 🔷 8. CREATE DATAFRAME
# =========================================
df = pd.DataFrame({
    # Demographic
    'Age': age,
    'Gender': gender,
    'Height_cm': height,
    'Weight_kg': weight,
    'BMI': bmi,

    # Lifestyle
    'Daily_Steps': daily_steps,
    'Activity_Level': activity_level,
    'Job_Type': job_type,
    'Smoking': smoking,
    'Alcohol': alcohol,
    'Sleep_Hours': sleep_hours,

    # Clinical
    'Systolic_BP': systolic_bp,
    'Diastolic_BP': diastolic_bp,
    'Fasting_Sugar': fasting_sugar,
    'HbA1c': hba1c,
    'Hemoglobin': hemoglobin,
    'Cholesterol': cholesterol,
    'Creatinine': creatinine,
    'Heart_Rate': heart_rate,
    'Visceral_Fat': visceral_fat,

    # Outputs (Probabilities)
    'Diabetes_Prob': diabetes_prob,
    'Hypertension_Prob': hypertension_prob,
    'Heart_Disease_Prob': heart_disease_prob,
    'Anemia_Prob': anemia_prob,
    'Kidney_Disease_Prob': kidney_disease_prob
})

# =========================================
# 🔷 9. DATA CLEANING (IMPORTANT)
# =========================================
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].clip(lower=0)  # remove negative values

# =========================================
# 🔷 10. SAVE DATASET
# =========================================
df.to_csv("multi_disease_dataset_v2.csv", index=False)

print("✅ Dataset created successfully!")
print("Shape:", df.shape)
print(df.head())

✅ Dataset created successfully!
Shape: (50000, 25)
   Age  Gender   Height_cm  Weight_kg        BMI  Daily_Steps Activity_Level  \
0   56  Female  171.731420  48.053862  16.294040         3163            Low   
1   69    Male  170.537294  72.545541  24.944338         7391       Moderate   
2   46  Female  177.344555  61.698069  19.617147        10732           High   
3   32    Male  175.142786  67.966828  22.157078        11806           High   
4   60  Female  179.350802  83.164683  25.854271         7168       Moderate   

    Job_Type  Smoking   Alcohol  ...  Hemoglobin  Cholesterol  Creatinine  \
0  Sedentary        1  Frequent  ...   11.180980   247.050420    1.548108   
1     Active        1      None  ...   12.238410   298.482640    1.609176   
2     Active        0      None  ...   11.368119   229.977725    1.405122   
3     Active        0      None  ...   12.018062   229.376228    1.156523   
4     Active        0  Frequent  ...   13.317089   241.964609    1.687331   

   He

In [24]:
df.shape

(50000, 25)

In [25]:
#So the Dataset is Ready And we will do the EDA, preprocessing and Model creation part in another JupyterNotebook